# Phase 2 — Gateway filter (developability MCDA)

Removes grid cells that are **legally or physically implausible** before resource/yield modeling.

| Input | Output |
|-------|--------|
| `data/processed/modeling_master_cell.parquet` | `data/processed/cells_passing_gateway.parquet` |
| | `maps/developability_ny.html` |

**Developability score `D`** (Phase 2 gateway):

```
D_raw = w_ag·(frac_81 + frac_82) + w_barren·frac_31 + w_open·frac_21
        - w_forest·(frac_41 + frac_42 + frac_43) - w_wet·(frac_90 + frac_95)
```

Normalized to **[0, 1]** via min–max across all cells.

**Hard filters:** water dominance, developed land, wetlands, low D, missing NLCD.

Map: cells clipped to the NY state polygon (`within` join), then Plotly mapbox (same center/zoom as the other screening maps).

In [ ]:
# --- User settings ---
D_MIN = 0.20
WATER_FRAC_MAX = 0.50
DEVELOPED_FRAC_MAX = 0.20      # sum frac_21..24
WETLAND_FRAC_MAX = 0.30        # sum frac_90 + frac_95
ICE_FRAC_MAX = 0.30
USE_POPULATION_FILTER = False  # unused; resource + seasonal screening lead
POP_DENSITY_MAX = 1000.0       # persons/km² (only if USE_POPULATION_FILTER)
NLCD_CELL_RADIUS_M = 7000      # matches notebook 02 zonal buffer; recorded in gateway report

# Developability weights (tune with domain input)
W_AG = 1.0
W_BARREN = 0.8
W_OPEN = 0.3
W_FOREST = 0.4
W_WET = 1.0

import warnings
warnings.filterwarnings('ignore')

import json
from pathlib import Path

import polars as pl
import yaml

ROOT = Path('../..').resolve()
CONFIG_PATH = ROOT / 'config' / 'paths.yaml'
MASTER_PATH = ROOT / 'data' / 'processed' / 'modeling_master_cell.parquet'
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'cells_passing_gateway.parquet'
GATEWAY_REPORT_PATH = ROOT / 'data' / 'processed' / 'gateway_filter_report.json'
MAP_PATH = ROOT / 'maps' / 'developability_ny.html'

with CONFIG_PATH.open(encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

if not MASTER_PATH.exists():
    raise FileNotFoundError(f'Run 05_build_master_table.ipynb first: {MASTER_PATH}')

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
MAP_PATH.parent.mkdir(parents=True, exist_ok=True)

master = pl.read_parquet(MASTER_PATH)
if 'in_ny_state' not in master.columns:
    raise ValueError('Re-run 05_build_master_table.ipynb — missing in_ny_state column')
print('Input cells:', len(master), '| in NY:', master['in_ny_state'].sum())

Input cells: 4401 | in NY: 1725


In [2]:
def frac(code: int) -> pl.Expr:
    return pl.col(f'frac_class_{code}').fill_null(0.0)


scored = master.with_columns([
    (frac(81) + frac(82)).alias('frac_agriculture'),
    frac(31).alias('frac_barren'),
    frac(21).alias('frac_dev_open'),
    (frac(41) + frac(42) + frac(43)).alias('frac_forest'),
    (frac(90) + frac(95)).alias('frac_wetland'),
    (frac(21) + frac(22) + frac(23) + frac(24)).alias('frac_developed'),
    frac(11).alias('frac_water'),
    frac(12).alias('frac_ice'),
]).with_columns(
    (
        W_AG * pl.col('frac_agriculture')
        + W_BARREN * pl.col('frac_barren')
        + W_OPEN * pl.col('frac_dev_open')
        - W_FOREST * pl.col('frac_forest')
        - W_WET * pl.col('frac_wetland')
    ).alias('developability_raw')
)

d_min = scored['developability_raw'].min()
d_max = scored['developability_raw'].max()
d_span = d_max - d_min if d_max != d_min else 1.0

scored = scored.with_columns(
    ((pl.col('developability_raw') - d_min) / d_span).alias('developability_D')
)

scored = scored.with_columns([
    (pl.col('frac_water') > WATER_FRAC_MAX).alias('fail_water'),
    (pl.col('frac_developed') > DEVELOPED_FRAC_MAX).alias('fail_developed'),
    (pl.col('frac_wetland') > WETLAND_FRAC_MAX).alias('fail_wetland'),
    (pl.col('frac_ice') > ICE_FRAC_MAX).alias('fail_ice'),
    (pl.col('developability_D') < D_MIN).alias('fail_low_D'),
    (pl.col('nlcd_pixel_count') == 0).alias('fail_no_nlcd'),
])

if USE_POPULATION_FILTER and 'population_density' in scored.columns:
    scored = scored.with_columns(
        (pl.col('population_density') > POP_DENSITY_MAX).alias('fail_population')
    )
else:
    scored = scored.with_columns(pl.lit(False).alias('fail_population'))

fail_cols = [c for c in scored.columns if c.startswith('fail_')]
scored = scored.with_columns(
    (~pl.any_horizontal([pl.col(c) for c in fail_cols])).alias('passes_gateway')
)

passing = scored.filter(pl.col('passes_gateway'))
passing.write_parquet(OUTPUT_PATH)
print(f'Wrote {len(passing):,} / {len(scored):,} cells -> {OUTPUT_PATH}')

Wrote 2,223 / 4,401 cells -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/cells_passing_gateway.parquet


In [3]:
# Filter breakdown + acceptance QA
report = {
    'input_cells': len(scored),
    'cells_passing': len(passing),
    'cells_excluded': len(scored) - len(passing),
    'pass_rate': round(len(passing) / len(scored), 4),
    'thresholds': {
        'D_MIN': D_MIN,
        'WATER_FRAC_MAX': WATER_FRAC_MAX,
        'DEVELOPED_FRAC_MAX': DEVELOPED_FRAC_MAX,
        'WETLAND_FRAC_MAX': WETLAND_FRAC_MAX,
        'ICE_FRAC_MAX': ICE_FRAC_MAX,
        'POP_DENSITY_MAX': POP_DENSITY_MAX,
    },
    'weights': {
        'W_AG': W_AG, 'W_BARREN': W_BARREN, 'W_OPEN': W_OPEN,
        'W_FOREST': W_FOREST, 'W_WET': W_WET,
    },
    'nlcd_r2': {
        'option': 'A',
        'cell_radius_m': NLCD_CELL_RADIUS_M,
        'v1_baseline_fail_no_nlcd': 721,
    },
    'fail_counts': {c: int(scored[c].sum()) for c in fail_cols},
    'water_dominant_cells': int(scored.filter(pl.col('frac_water') > WATER_FRAC_MAX).height),
    'water_dominant_excluded': int(scored.filter(pl.col('fail_water')).height),
    'developability_D_mean_passing': float(passing['developability_D'].mean()) if len(passing) else None,
    'developability_D_mean_all': float(scored['developability_D'].mean()),
    'cells_in_ny': int(scored.filter(pl.col('in_ny_state')).height) if 'in_ny_state' in scored.columns else None,
    'cells_passing_in_ny': int(scored.filter(pl.col('passes_gateway') & pl.col('in_ny_state')).height) if 'in_ny_state' in scored.columns else None,
}

GATEWAY_REPORT_PATH.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))

scored.select([
    pl.len().alias('n_cells'),
    pl.col('passes_gateway').sum().alias('n_passing'),
    pl.col('developability_D').mean().alias('mean_D_all'),
    pl.col('developability_D').filter(pl.col('passes_gateway')).mean().alias('mean_D_passing'),
])

{
  "input_cells": 4401,
  "cells_passing": 2223,
  "cells_excluded": 2178,
  "pass_rate": 0.5051,
  "thresholds": {
    "D_MIN": 0.2,
    "WATER_FRAC_MAX": 0.5,
    "DEVELOPED_FRAC_MAX": 0.2,
    "WETLAND_FRAC_MAX": 0.3,
    "ICE_FRAC_MAX": 0.3,
    "POP_DENSITY_MAX": 1000.0
  },
  "weights": {
    "W_AG": 1.0,
    "W_BARREN": 0.8,
    "W_OPEN": 0.3,
    "W_FOREST": 0.4,
    "W_WET": 1.0
  },
  "nlcd_r2": {
    "option": "A",
    "cell_radius_m": 7000,
    "v1_baseline_fail_no_nlcd": 721
  },
  "fail_counts": {
    "fail_water": 316,
    "fail_developed": 513,
    "fail_wetland": 23,
    "fail_ice": 0,
    "fail_low_D": 658,
    "fail_no_nlcd": 703,
    "fail_population": 0
  },
  "water_dominant_cells": 316,
  "water_dominant_excluded": 316,
  "developability_D_mean_passing": 0.42939457224684513,
  "developability_D_mean_all": 0.39553724684449765,
  "cells_in_ny": 1725,
  "cells_passing_in_ny": 1089
}


n_cells,n_passing,mean_D_all,mean_D_passing
u32,u32,f64,f64
4401,2223,0.395537,0.429395


In [4]:
# Plotly map — clip cells to NY state boundary, then plot (same framing as other screening maps)
import geopandas as gpd
import pandas as pd
import plotly.graph_objects as go

NY_STATES_URL = (
    'https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json'
)

map_df = scored.select([
    'latitude', 'longitude', 'developability_D', 'passes_gateway',
]).to_pandas()

gdf = gpd.GeoDataFrame(
    map_df,
    geometry=gpd.points_from_xy(map_df['longitude'], map_df['latitude']),
    crs='EPSG:4326',
)
print(f'Grid cells before NY clip: {len(gdf):,}')

ny_boundary = gpd.read_file(NY_STATES_URL)
ny_boundary = ny_boundary[ny_boundary['name'] == 'New York'].copy()
clipped = gpd.sjoin(gdf, ny_boundary, predicate='within', how='inner')
drop_cols = [c for c in clipped.columns if c in ('index_right', 'name', 'geometry')]
clipped = pd.DataFrame(clipped.drop(columns=drop_cols, errors='ignore'))
print(f'Grid cells inside NY: {len(clipped):,}')

clipped['status'] = clipped['passes_gateway'].map(
    {True: 'Passes gateway', False: 'Excluded'}
)
clipped['marker_opacity'] = (0.35 + 0.55 * clipped['developability_D']).clip(upper=0.95)
clipped['hover'] = clipped.apply(
    lambda r: (
        f"lat={r['latitude']:.2f}, lon={r['longitude']:.2f}<br>"
        f"D={r['developability_D']:.3f}<br>"
        f"pass={'yes' if r['passes_gateway'] else 'no'}"
    ),
    axis=1,
)

fig = go.Figure()
for passes, label, color in [
    (False, 'Excluded', '#d62728'),
    (True, 'Passes gateway', '#2ca02c'),
]:
    sub = clipped[clipped['passes_gateway'] == passes]
    if sub.empty:
        continue
    fig.add_trace(
        go.Scattermap(
            lat=sub['latitude'],
            lon=sub['longitude'],
            mode='markers',
            name=label,
            marker=dict(size=9, color=color, opacity=sub['marker_opacity'].tolist()),
            text=sub['hover'],
            hoverinfo='text',
        )
    )

# NY state outline so the map reads as NY-shaped, not a rectangle of dots
ny_outline = ny_boundary.boundary.iloc[0]
outline_lons, outline_lats = ny_outline.xy
fig.add_trace(
    go.Scattermap(
        lat=list(outline_lats),
        lon=list(outline_lons),
        mode='lines',
        name='NY border',
        line=dict(width=2, color='#333333'),
        hoverinfo='skip',
        showlegend=False,
    )
)

fig.update_layout(
    map=dict(
        style='carto-positron',
        center=dict(lat=42.9, lon=-75.8),
        zoom=5.5,
    ),
    title='Developability gateway (NY state boundary)',
    margin=dict(r=0, t=40, l=0, b=0),
    height=800,
    legend=dict(title='Gateway status'),
)

fig.write_html(str(MAP_PATH), include_plotlyjs='cdn')
print(f'Map -> {MAP_PATH}')

Grid cells before NY clip: 4,401


Grid cells inside NY: 1,725


Map -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/maps/developability_ny.html


In [5]:
passing.sort('developability_D', descending=True).head(10)

latitude,longitude,shortwave_mean,shortwave_mean_24h,shortwave_interannual_std,shortwave_intraseasonal_std_mean,direct_mean,diffuse_mean,shortwave_diurnal_amplitude,clear_sky_index,direct_diffuse_ratio,wind_speed_mean,wind_speed_interannual_std,wind_speed_intraseasonal_std_mean,wind_speed_p90,wind_power_mean,wind_speed_hub_mean,wind_speed_hub_std,wind_hub_power_mean,wind_hub_speed_p50,wind_hub_speed_p75,wind_hub_speed_p90,wind_hub_speed_p95,wind_ramp_p95,weibull_k,weibull_A,temperature_mean,surface_pressure_mean,n_years,year_start,year_end,shortwave_mean_m01,shortwave_mean_m02,shortwave_mean_m03,shortwave_mean_m04,shortwave_mean_m05,shortwave_mean_m06,…,nlcd_pixel_count,frac_class_11,frac_class_12,frac_class_21,frac_class_22,frac_class_23,frac_class_24,frac_class_31,frac_class_41,frac_class_42,frac_class_43,frac_class_52,frac_class_71,frac_class_81,frac_class_82,frac_class_90,frac_class_95,dist_to_transmission_km,in_ny_state,frac_agriculture,frac_barren,frac_dev_open,frac_forest,frac_wetland,frac_developed,frac_water,frac_ice,developability_raw,developability_D,fail_water,fail_developed,fail_wetland,fail_ice,fail_low_D,fail_no_nlcd,fail_population,passes_gateway
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,i32,i32,f64,f64,f64,f64,f64,f64,…,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool
42.84,-76.83,293.801029,157.023389,9.815234,235.918884,96.095488,60.927902,972.875,0.442875,1.55726,3.195413,0.230561,1.689208,5.58326,64.851558,4.275231,2.260038,155.31672,3.876896,5.681078,7.469997,8.53151,1.75104,1.985058,4.821241,10.308785,996.056372,8,2018,2025,148.455509,210.322432,282.344263,316.603557,366.670129,391.773196,…,170769,0.001745,0.0,0.035803,0.056451,0.006652,0.000644,0.005089,0.057955,0.001634,0.003121,0.000029,0.000211,0.102003,0.677588,0.049336,0.001739,10.982107,true,0.779591,0.005089,0.035803,0.06271,0.051075,0.09955,0.001745,0.0,0.718244,1.0,false,false,false,false,false,false,false,true
42.84,-77.91,295.261057,157.515945,9.511062,236.134225,96.346265,61.16968,973.25,0.444553,1.581057,3.318978,0.211756,1.691678,5.668625,69.850493,4.440552,2.263342,167.288956,4.090926,5.791685,7.58421,8.67434,1.739516,2.068422,5.010605,9.914926,985.758552,8,2018,2025,142.406725,211.768469,283.691864,321.110684,368.191815,396.030641,…,170771,0.005352,0.0,0.035094,0.043257,0.006968,0.00113,0.000006,0.148538,0.00178,0.004767,0.000018,0.00024,0.114545,0.603369,0.032956,0.001979,2.140589,true,0.717915,0.000006,0.035094,0.155085,0.034936,0.086449,0.005352,0.0,0.631478,0.934194,false,false,false,false,false,false,false,true
41.04,-76.74,310.600531,165.792211,14.63311,242.979093,104.858129,60.934082,977.75,0.464527,1.733318,2.713707,0.131702,1.503622,4.788222,41.994181,3.630743,2.011738,100.574275,3.250411,4.767748,6.406294,7.488003,1.757793,1.882438,4.088139,11.394483,994.810213,8,2018,2025,184.633291,245.711924,311.255929,341.036886,364.558505,392.45651,…,170768,0.00072,0.0,0.066558,0.058571,0.013445,0.003268,0.004292,0.144518,0.000422,0.015647,0.000996,0.000533,0.19653,0.484119,0.009615,0.000767,3.102174,false,0.680649,0.004292,0.066558,0.160586,0.010383,0.141842,0.00072,0.0,0.629433,0.932643,false,false,false,false,false,false,false,true
42.75,-77.1,295.704301,158.056044,9.913685,236.258853,96.270125,61.78592,974.875,0.445769,1.55163,3.587199,0.287304,1.77146,6.04036,85.799993,4.799413,2.370085,205.487328,4.51084,6.232396,8.081564,9.233081,1.689774,2.147423,5.416522,9.67327,980.809746,8,2018,2025,148.855605,211.688241,285.895064,320.723122,366.608621,393.247533,…,170757,0.000135,0.0,0.034821,0.047079,0.004533,0.000322,0.000111,0.112077,0.001909,0.006459,0.000041,0.000035,0.066252,0.653718,0.07181,0.000697,7.450959,true,0.71997,0.000111,0.034821,0.120446,0.072507,0.086755,0.000135,0.0,0.609821,0.917769,false,false,false,false,false,false,false,true
42.75,-76.56,295